In [4]:
import os
from pathlib import Path

# Define base paths for the ASVspoof 2021 dataset
BASE_DIR = Path(r"e:/Research Hub/ASVDeepFake")

# Audio file paths
AUDIO_DIR = BASE_DIR  / "ASVspoof2021_DF_eval" / "flac"

# Trial and metadata paths
TRIAL_FILE = BASE_DIR / "ASVspoof2021_DF_eval" / "ASVspoof2021.DF.cm.eval.trl.txt"
METADATA_FILE = BASE_DIR / "DF-keys-full" / "keys" / "DF" / "CM" / "trial_metadata.txt"

# Baseline system paths
BASELINE_DIR = BASE_DIR / "DF-keys-full" / "keys" / "DF" / "CM"
CQCC_GMM_DIR = BASELINE_DIR / "CQCC-GMM"
LFCC_GMM_DIR = BASELINE_DIR / "LFCC-GMM"
LFCC_LCNN_DIR = BASELINE_DIR / "LFCC-LCNN"
RAWNET2_DIR = BASELINE_DIR / "RawNet2"

# Verify paths exist
print("Checking dataset paths...")
print(f"Base directory: {BASE_DIR.exists()} - {BASE_DIR}")
print(f"Audio directory: {AUDIO_DIR.exists()} - {AUDIO_DIR}")
print(f"Trial file: {TRIAL_FILE.exists()} - {TRIAL_FILE}")
print(f"Metadata file: {METADATA_FILE.exists()} - {METADATA_FILE}")

# Count audio files
if AUDIO_DIR.exists():
    audio_files = list(AUDIO_DIR.glob("*.flac"))
    print(f"\nFound {len(audio_files)} FLAC audio files")
    
    # Show first few filenames as examples
    print("\nFirst 5 audio files:")
    for i, file in enumerate(audio_files[:5]):
        print(f"  {i+1}. {file.name}")
else:
    print("Audio directory not found!")

Checking dataset paths...
Base directory: True - e:\Research Hub\ASVDeepFake
Audio directory: True - e:\Research Hub\ASVDeepFake\ASVspoof2021_DF_eval\flac
Trial file: True - e:\Research Hub\ASVDeepFake\ASVspoof2021_DF_eval\ASVspoof2021.DF.cm.eval.trl.txt
Metadata file: True - e:\Research Hub\ASVDeepFake\DF-keys-full\keys\DF\CM\trial_metadata.txt

Found 611829 FLAC audio files

First 5 audio files:
  1. DF_E_2000011.flac
  2. DF_E_2000013.flac
  3. DF_E_2000024.flac
  4. DF_E_2000026.flac
  5. DF_E_2000027.flac


In [5]:
import pandas as pd

print("Loading metadata and counting real vs fake files...")
print("=" * 50)

# Load the metadata file (contains the ground truth labels)
try:
    # Read metadata file - let's first examine its structure
    with open(METADATA_FILE, 'r') as f:
        sample_lines = [f.readline().strip() for _ in range(5)]
    
    print("Sample metadata lines:")
    for i, line in enumerate(sample_lines):
        print(f"  {i+1}. {line}")
    
    # Parse the metadata file properly
    with open(METADATA_FILE, 'r') as f:
        lines = f.readlines()
    
    # Parse each line manually to handle variable number of columns
    parsed_data = []
    for line in lines:
        parts = line.strip().split()
        if len(parts) >= 4:  # Ensure minimum required columns
            speaker_id = parts[0]
            audio_filename = parts[1]
            # Find the label (bonafide or spoof)
            if 'bonafide' in parts:
                label = 'bonafide'
                system_id = '-'
            elif 'spoof' in parts:
                label = 'spoof'
                # Extract system information
                system_id = parts[2] if len(parts) > 2 else 'unknown'
            else:
                continue
            
            parsed_data.append({
                'speaker_id': speaker_id,
                'audio_filename': audio_filename,
                'system_id': system_id,
                'label': label
            })
    
    metadata_df = pd.DataFrame(parsed_data)
    
    print(f"\nSuccessfully parsed {len(metadata_df):,} metadata entries")
    
    # Count real vs fake files
    print(f"\n📊 REAL vs FAKE COUNT")
    print("=" * 30)
    
    label_counts = metadata_df['label'].value_counts()
    total_files = len(metadata_df)
    
    real_count = label_counts.get('bonafide', 0)
    fake_count = label_counts.get('spoof', 0)
    
    print(f"Total files in metadata: {total_files:,}")
    print(f"Real (bonafide) files:   {real_count:,} ({real_count/total_files*100:.1f}%)")
    print(f"Fake (spoof) files:      {fake_count:,} ({fake_count/total_files*100:.1f}%)")
    
    # Calculate ratio
    if real_count > 0:
        ratio = fake_count / real_count
        print(f"Fake-to-Real ratio:      {ratio:.1f}:1")
    
    # Show examples of each type
    print(f"\n📋 Examples of REAL (bonafide) files:")
    real_files = metadata_df[metadata_df['label'] == 'bonafide']
    for i, (_, row) in enumerate(real_files.head(5).iterrows()):
        print(f"  {i+1}. {row['audio_filename']} - Speaker: {row['speaker_id']}")
    
    print(f"\n📋 Examples of FAKE (spoof) files:")
    fake_files = metadata_df[metadata_df['label'] == 'spoof']
    for i, (_, row) in enumerate(fake_files.head(5).iterrows()):
        print(f"  {i+1}. {row['audio_filename']} - Speaker: {row['speaker_id']}, System: {row['system_id']}")
    
    # Show deepfake systems
    spoof_systems = fake_files['system_id'].unique()
    print(f"\n🔧 Deepfake generation systems: {len(spoof_systems)}")
    print("System distribution:")
    system_counts = fake_files['system_id'].value_counts()
    for system, count in system_counts.head(10).items():
        print(f"  {system}: {count:,} files")
    
    # Verify the count matches our audio files
    print(f"\n✅ Verification:")
    print(f"Audio files found: {len(audio_files):,}")
    print(f"Metadata entries:  {len(metadata_df):,}")
    
    if len(audio_files) == len(metadata_df):
        print("✅ Perfect match! All audio files have metadata entries.")
    else:
        print(f"⚠️  Mismatch detected: {abs(len(audio_files) - len(metadata_df))} difference")

except FileNotFoundError:
    print(f"❌ Metadata file not found: {METADATA_FILE}")
except Exception as e:
    print(f"❌ Error reading metadata: {e}")
    import traceback
    traceback.print_exc()

Loading metadata and counting real vs fake files...
Sample metadata lines:
  1. LA_0023 DF_E_2000011 nocodec asvspoof A14 spoof notrim progress traditional_vocoder - - - -
  2. TEF2 DF_E_2000013 low_m4a vcc2020 Task1-team20 spoof notrim eval neural_vocoder_nonautoregressive Task1 team20 FF E
  3. TGF1 DF_E_2000024 mp3m4a vcc2020 Task2-team12 spoof notrim eval traditional_vocoder Task2 team12 FF G
  4. LA_0043 DF_E_2000026 mp3m4a asvspoof A09 spoof notrim eval traditional_vocoder - - - -
  5. LA_0021 DF_E_2000027 mp3m4a asvspoof A12 spoof notrim eval neural_vocoder_autoregressive - - - -

Successfully parsed 611,829 metadata entries

📊 REAL vs FAKE COUNT
Total files in metadata: 611,829
Real (bonafide) files:   22,617 (3.7%)
Fake (spoof) files:      589,212 (96.3%)
Fake-to-Real ratio:      26.1:1

📋 Examples of REAL (bonafide) files:
  1. DF_E_2000053 - Speaker: VCC2TM2
  2. DF_E_2000058 - Speaker: LA_0048
  3. DF_E_2000079 - Speaker: TEM2
  4. DF_E_2000246 - Speaker: LA_0004
  5. DF_E_

In [ ]:
# Create convenient variables for real and fake files
print("\n🔧 Creating convenient variables...")

# Separate real and fake files
real_files = metadata_df[metadata_df['label'] == 'bonafide'].copy()
fake_files = metadata_df[metadata_df['label'] == 'spoof'].copy()

print(f"✅ real_files dataset: {len(real_files):,} entries")
print(f"✅ fake_files dataset: {len(fake_files):,} entries")

# Function to check if a specific audio file is real or fake
def check_file_authenticity(filename, metadata_df=metadata_df):
    """
    Check if an audio file is real (bonafide) or fake (spoof)
    
    Args:
        filename (str): Audio filename (with or without .flac extension)
        metadata_df (DataFrame): Metadata dataframe to search in
    
    Returns:
        dict: Information about the file including label, speaker, system
    """
    # Remove .flac extension if present for matching
    clean_filename = filename.replace('.flac', '')
    
    result = metadata_df[metadata_df['audio_filename'] == clean_filename]
    if not result.empty:
        row = result.iloc[0]
        return {
            'filename': filename,
            'label': row['label'],
            'is_real': row['label'] == 'bonafide',
            'speaker_id': row['speaker_id'],
            'system_id': row['system_id'] if row['label'] == 'spoof' else None,
            'authentic': 'REAL' if row['label'] == 'bonafide' else 'FAKE'
        }
    return None

# Test the function with some examples
print(f"\n🧪 Testing file authenticity checker:")

# Test with a real file
real_example = real_files.iloc[0]['audio_filename']
result = check_file_authenticity(real_example)
print(f"Real file test: {result}")

# Test with a fake file  
fake_example = fake_files.iloc[0]['audio_filename']
result = check_file_authenticity(fake_example)
print(f"Fake file test: {result}")

# Test with .flac extension
result = check_file_authenticity(fake_example + '.flac')
print(f"With .flac extension: {result}")

print(f"\n✅ Setup complete! You can now use:")
print(f"   - metadata_df: Full dataset ({len(metadata_df):,} files)")
print(f"   - real_files: Real audio only ({len(real_files):,} files)")  
print(f"   - fake_files: Fake audio only ({len(fake_files):,} files)")
print(f"   - check_file_authenticity(filename): Check any specific file")


🔧 Creating convenient variables...
✅ real_files dataset: 22,617 entries
✅ fake_files dataset: 589,212 entries

🧪 Testing file authenticity checker:
Real file test: {'filename': 'DF_E_2000053', 'label': 'bonafide', 'is_real': True, 'speaker_id': 'VCC2TM2', 'system_id': None, 'authentic': 'REAL'}
Fake file test: {'filename': 'DF_E_2000011', 'label': 'spoof', 'is_real': False, 'speaker_id': 'LA_0023', 'system_id': 'nocodec', 'authentic': 'FAKE'}
With .flac extension: {'filename': 'DF_E_2000011.flac', 'label': 'spoof', 'is_real': False, 'speaker_id': 'LA_0023', 'system_id': 'nocodec', 'authentic': 'FAKE'}

✅ Setup complete! You can now use:
   - metadata_df: Full dataset (611,829 files)
   - real_files: Real audio only (22,617 files)
   - fake_files: Fake audio only (589,212 files)
   - check_file_authenticity(filename): Check any specific file


: 